# Support Vector Machines — Soft Margin and SMO Algorithm

## Learning Objectives
- Explain why the hard-margin SVM fails on non-linearly separable data and with outliers
- Introduce **slack variables** $\\xi_i$ to allow controlled margin violations
- Formulate the **soft-margin SVM** primal and dual, and explain the role of $C$
- State the **KKT conditions** for the soft-margin case and classify points by $\\alpha_i$ and $\\xi_i$
- Describe the **SMO algorithm** — coordinate ascent on the dual $\\alpha$ variables
- Derive the **closed-form update** for a pair $(\\alpha_i, \\alpha_j)$
- Explain the **heuristics** for choosing which pair to update and the KKT convergence test

## 1. The Hard-Margin Problem

The SVM derived earlier assumed the data is **linearly separable** — there exists a
hyperplane that perfectly separates the two classes. Two practical problems arise:

| Problem | Effect |
|---|---|
| **Non-separable data** | No feasible solution exists; the primal QP is infeasible |
| **Outliers** | A single outlier can drastically shift the margin boundary |

The figure below shows how one outlier collapses the hard-margin boundary:
the margin shrinks dramatically and the hyperplane tilts to accommodate the outlier,
harming generalisation on the remaining data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC

rng = np.random.default_rng(5)
pos = rng.normal(loc=[4.0, 4.0], scale=0.4, size=(12, 2))
neg = rng.normal(loc=[1.5, 1.5], scale=0.4, size=(12, 2))

# add a single outlier in the positive class region but negative label
outlier = np.array([[3.4, 3.3]])

X_clean = np.vstack([pos, neg])
y_clean = np.hstack([np.ones(12), -np.ones(12)])
X_out   = np.vstack([pos, neg, outlier])
y_out   = np.hstack([np.ones(12), -np.ones(12), -1])

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, X, y, title in [
    (axes[0], X_clean, y_clean, 'Hard-margin SVM — clean data'),
    (axes[1], X_out,   y_out,   'Hard-margin SVM — one outlier collapses margin'),
]:
    clf = SVC(kernel='linear', C=1e8)
    clf.fit(X, y)
    w, b = clf.coef_[0], clf.intercept_[0]
    xr = np.linspace(-0.5, 5.5, 300)
    ax.plot(xr, -(w[0]*xr+b)/w[1], 'k-', lw=2.2, label='boundary')
    ax.plot(xr, -(w[0]*xr+b-1)/w[1], 'b--', lw=1.5)
    ax.plot(xr, -(w[0]*xr+b+1)/w[1], 'r--', lw=1.5)
    ax.fill_between(xr, -(w[0]*xr+b-1)/w[1], -(w[0]*xr+b+1)/w[1],
                    alpha=0.10, color='#4dac26')
    ax.scatter(X[y== 1,0], X[y== 1,1], marker='x', s=80, c='#2166ac',
               linewidths=2, label='$y=+1$')
    ax.scatter(X[y==-1,0], X[y==-1,1], marker='o', s=65, facecolors='none',
               edgecolors='#d6604d', linewidths=2, label='$y=-1$')
    sv = clf.support_vectors_
    ax.scatter(sv[:,0], sv[:,1], s=260, facecolors='none',
               edgecolors='#ff7f00', linewidths=2.2, zorder=5)
    mw = 2.0/np.linalg.norm(w)
    ax.text(0.03, 0.03, f'Margin = {mw:.3f}  |  SVs = {len(sv)}',
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))
    ax.set_xlim(-0.5, 5.5); ax.set_ylim(-0.5, 5.5)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9, loc='upper left')

# mark outlier
axes[1].scatter(*outlier.T, s=250, marker='D', c='#d6604d',
                zorder=7, label='outlier')
axes[1].legend(fontsize=9, loc='upper left')

fig.suptitle('Hard-Margin Failure: One Outlier Collapses the Margin', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Soft-Margin SVM

### 2.1 Slack variables

To allow controlled margin violations, introduce **slack variables** $\xi_i \geq 0$:

- $\xi_i = 0$: point $i$ is correctly classified with margin $\geq 1$
- $0 < \xi_i \leq 1$: point $i$ is correctly classified but inside the margin
- $\xi_i > 1$: point $i$ is **misclassified**

The soft-margin constraint relaxes $y^{(i)}(w^T x^{(i)} + b) \geq 1$ to:

$$y^{(i)}(w^T x^{(i)} + b) \geq 1 - \xi_i$$

### 2.2 Soft-margin primal QP

$$\boxed{\min_{w,\,b,\,\xi}\; \frac{1}{2}\|w\|^2 + C\sum_{i=1}^n \xi_i}$$

$$\text{s.t.} \quad y^{(i)}(w^T x^{(i)} + b) \geq 1 - \xi_i,\quad \xi_i \geq 0,\quad i=1,\ldots,n$$

### 2.3 The role of $C$

| Value of $C$ | Effect | Risk |
|---|---|---|
| Large $C$ | Penalises violations heavily → narrow/hard margin | Overfitting |
| Small $C$ | Tolerates violations → wide/soft margin | Underfitting |
| $C \to \infty$ | Recovers the hard-margin SVM | Infeasible if not separable |

$C$ is the key **regularisation hyperparameter** of the SVM — chosen by cross-validation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC

rng = np.random.default_rng(5)
pos = rng.normal(loc=[4.0, 4.0], scale=0.5, size=(14, 2))
neg = rng.normal(loc=[1.5, 1.5], scale=0.5, size=(14, 2))
outlier = np.array([[3.4, 3.3]])
X = np.vstack([pos, neg, outlier])
y = np.hstack([np.ones(14), -np.ones(14), -1])

C_vals = [0.1, 1.0, 100.0]
fig, axes = plt.subplots(1, 3, figsize=(15, 5.0))

for ax, C in zip(axes, C_vals):
    clf = SVC(kernel='linear', C=C)
    clf.fit(X, y)
    w, b = clf.coef_[0], clf.intercept_[0]
    xr = np.linspace(-0.5, 5.5, 300)
    ax.plot(xr, -(w[0]*xr+b)/w[1], 'k-', lw=2.2)
    ax.plot(xr, -(w[0]*xr+b-1)/w[1], 'b--', lw=1.5)
    ax.plot(xr, -(w[0]*xr+b+1)/w[1], 'r--', lw=1.5)
    ax.fill_between(xr, -(w[0]*xr+b-1)/w[1], -(w[0]*xr+b+1)/w[1],
                    alpha=0.10, color='#4dac26')
    ax.scatter(X[y== 1,0], X[y== 1,1], marker='x', s=70, c='#2166ac', linewidths=2)
    ax.scatter(X[y==-1,0], X[y==-1,1], marker='o', s=55, facecolors='none',
               edgecolors='#d6604d', linewidths=2)
    ax.scatter(*outlier.T, s=220, marker='D', c='#d6604d', zorder=7)
    sv = clf.support_vectors_
    ax.scatter(sv[:,0], sv[:,1], s=240, facecolors='none',
               edgecolors='#ff7f00', linewidths=2.2, zorder=5)
    mw = 2.0/np.linalg.norm(w)
    ax.set_xlim(-0.5, 5.5); ax.set_ylim(-0.5, 5.5)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_title(f'$C = {C}$', fontsize=12, fontweight='bold')
    ax.text(0.03, 0.03, f'Margin={mw:.2f}  SVs={len(sv)}',
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

fig.suptitle('Soft-Margin SVM: Effect of $C$ on Margin Width and Tolerance to Violations',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Soft-Margin Dual

### 3.1 Lagrangian

Introducing multipliers $\alpha_i \geq 0$ for the margin constraints and $r_i \geq 0$ for $\xi_i \geq 0$:

$$\mathcal{L}(w, b, \xi, \alpha, r) = \frac{1}{2}\|w\|^2 + C\sum_i \xi_i - \sum_i \alpha_i[y^{(i)}(w^Tx^{(i)}+b)-1+\xi_i] - \sum_i r_i \xi_i$$

Setting $\partial \mathcal{L}/\partial \xi_i = 0$ gives $C - \alpha_i - r_i = 0$, so $r_i = C - \alpha_i \geq 0$,
which means $\alpha_i \leq C$.

### 3.2 Soft-margin dual QP

The dual has the **same form** as the hard-margin dual — only the constraint on $\alpha_i$ changes:

$$\boxed{\max_{\alpha}\; W(\alpha) = \sum_{i=1}^n \alpha_i - \frac{1}{2}\sum_{i,j} y^{(i)} y^{(j)} \alpha_i \alpha_j \langle x^{(i)}, x^{(j)} \rangle}$$

$$\text{s.t.} \quad \mathbf{0 \leq \alpha_i \leq C},\quad \sum_{i=1}^n \alpha_i y^{(i)} = 0$$

The only change: $\alpha_i \geq 0$ becomes $0 \leq \alpha_i \leq C$.

### 3.3 KKT conditions for soft-margin

The KKT dual-complementarity conditions partition training points into three categories:

| Condition | Interpretation |
|---|---|
| $\alpha_i = 0$ | Correctly classified, outside margin ($\xi_i = 0$) |
| $0 < \alpha_i < C$ | **Support vector on the margin boundary** ($\xi_i = 0$, functional margin = 1) |
| $\alpha_i = C$ | Support vector inside margin or misclassified ($\xi_i > 0$) |

The KKT conditions also give the convergence criterion for SMO (see §5).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC

rng = np.random.default_rng(99)
pos = rng.normal(loc=[3.8, 3.8], scale=0.55, size=(15, 2))
neg = rng.normal(loc=[1.4, 1.4], scale=0.55, size=(15, 2))
# add a few overlapping points
overlap = np.array([[2.8, 2.8], [2.5, 2.5], [2.3, 2.6]])
X = np.vstack([pos, neg, overlap, overlap + 0.3])
y = np.hstack([np.ones(15), -np.ones(15), -np.ones(3), np.ones(3)])

clf = SVC(kernel='linear', C=1.0)
clf.fit(X, y)
w, b = clf.coef_[0], clf.intercept_[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# ── Left: colour-coded point categories ──────────────────────────────────
ax = axes[0]
xr = np.linspace(-0.5, 5.5, 300)
ax.plot(xr, -(w[0]*xr+b)/w[1], 'k-', lw=2.2, label='boundary', zorder=3)
ax.plot(xr, -(w[0]*xr+b-1)/w[1], 'b--', lw=1.5, label='$w^Tx+b=+1$', zorder=3)
ax.plot(xr, -(w[0]*xr+b+1)/w[1], 'r--', lw=1.5, label='$w^Tx+b=-1$', zorder=3)
ax.fill_between(xr, -(w[0]*xr+b-1)/w[1], -(w[0]*xr+b+1)/w[1], alpha=0.10, color='#4dac26')

margins = y * (X @ w + b)
for i, (xi, mi) in enumerate(zip(X, margins)):
    if mi >= 1 - 1e-4:          # alpha_i = 0, outside margin
        c, m, s, z = '#aaaaaa', 'x' if y[i]>0 else 'o', 55, 3
    elif abs(mi - 1) < 1e-4:    # 0 < alpha_i < C, on margin
        c, m, s, z = '#ff7f00', 'x' if y[i]>0 else 'o', 120, 6
    else:                        # alpha_i = C, inside/wrong side
        c, m, s, z = '#d6604d' if y[i]<0 else '#2166ac', '^', 110, 5
    if m == 'o':
        ax.scatter(xi[0], xi[1], s=s, marker=m, facecolors='none', edgecolors=c, linewidths=2, zorder=z)
    else:
        ax.scatter(xi[0], xi[1], s=s, marker=m, c=c, linewidths=2, zorder=z)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker='x', color='w', markerfacecolor='#aaaaaa', markersize=9, markeredgewidth=2, label='$\\alpha_i=0$ (outside margin)'),
    Line2D([0],[0], marker='x', color='w', markerfacecolor='#ff7f00', markersize=9, markeredgewidth=2, label='$0<\\alpha_i<C$ (on margin, SV)'),
    Line2D([0],[0], marker='^', color='w', markerfacecolor='#d6604d', markersize=9, label='$\\alpha_i=C$ (margin violation)'),
]
ax.legend(handles=legend_elements, fontsize=8.5, loc='upper left')
ax.set_xlim(-0.5, 5.5); ax.set_ylim(-0.5, 5.5)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('KKT Point Categories ($C=1$)', fontsize=11)

# ── Right: C sweep — n_SVs and train accuracy ────────────────────────────
ax = axes[1]
C_vals = np.logspace(-2, 3, 30)
n_svs = []
accs  = []
for C in C_vals:
    clf_c = SVC(kernel='linear', C=C)
    clf_c.fit(X, y)
    n_svs.append(len(clf_c.support_vectors_))
    accs.append((clf_c.predict(X) == y).mean())

ax2 = ax.twinx()
ax.semilogx(C_vals, n_svs, 'b-o', ms=4, lw=2, label='# support vectors')
ax2.semilogx(C_vals, accs, 'r-s', ms=4, lw=2, label='train accuracy')
ax.set_xlabel('$C$ (regularisation)', fontsize=11)
ax.set_ylabel('# support vectors', color='blue', fontsize=10)
ax2.set_ylabel('Training accuracy', color='red', fontsize=10)
ax.set_title('Effect of $C$: Support Vectors and Accuracy', fontsize=11)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=9)

fig.suptitle('Soft-Margin SVM: KKT Categories and C Trade-off', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. The SMO Algorithm

### 4.1 The challenge of solving the dual QP

The soft-margin dual has $n$ variables $\alpha_1,\ldots,\alpha_n$ with the equality constraint
$\sum_i \alpha_i y^{(i)} = 0$. Generic QP solvers scale as $O(n^3)$ and require $O(n^2)$ memory
to store the kernel matrix — infeasible for large $n$.

### 4.2 Coordinate ascent — why a single variable fails

Coordinate ascent would update one $\alpha_i$ at a time while holding all others fixed. But the
equality constraint $\sum_i \alpha_i y^{(i)} = 0$ means **changing any single $\alpha_i$ violates
the constraint** — we cannot optimise over a single variable.

### 4.3 SMO: optimise a pair at a time

**Sequential Minimal Optimization** (John Platt, 1998) selects the **smallest non-trivial
sub-problem**: update a pair $(\alpha_i, \alpha_j)$ simultaneously. With two variables and
the constraint $\alpha_i y^{(i)} + \alpha_j y^{(j)} = \text{const}$, the pair can be eliminated
to a **1-D constrained quadratic** — which has a closed-form solution.

**Main loop:**
```
repeat:
    1. Select a pair (alpha_i, alpha_j) to update  [heuristic]
    2. Compute the optimal (alpha_i, alpha_j) analytically  [closed form]
    3. Update b  [using KKT conditions]
until KKT conditions are satisfied to within tolerance
```

## 5. SMO Closed-Form Update

### 5.1 The 1-D subproblem

Fix all $\alpha_k$ except $\alpha_i$ and $\alpha_j$. The constraint becomes:

$$\alpha_i y^{(i)} + \alpha_j y^{(j)} = -\sum_{k \neq i,j} \alpha_k y^{(k)} = \text{const} \triangleq \zeta$$

So $\alpha_i = (\zeta - \alpha_j y^{(j)}) y^{(i)}$. Substituting into $W(\alpha)$ gives a univariate
quadratic in $\alpha_j$ — maximised analytically.

### 5.2 Unconstrained update

Define the **error** for example $i$:

$$E_i = h(x^{(i)}) - y^{(i)} = \left(\sum_k \alpha_k y^{(k)} K(x^{(k)}, x^{(i)}) + b\right) - y^{(i)}$$

The unconstrained optimum for $\alpha_j$ is:

$$\alpha_j^{\text{new}} = \alpha_j^{\text{old}} - \frac{y^{(j)}(E_i - E_j)}{\eta}$$

where $\eta = K_{ii} + K_{jj} - 2K_{ij} = \|\phi(x^{(i)}) - \phi(x^{(j)})\|^2 \geq 0$.

### 5.3 Clip to the box $[L, H]$

The box bounds ensure $0 \leq \alpha_i, \alpha_j \leq C$:

| Case | $L$ | $H$ |
|---|---|---|
| $y^{(i)} \neq y^{(j)}$ | $\max(0, \alpha_j - \alpha_i)$ | $\min(C, C + \alpha_j - \alpha_i)$ |
| $y^{(i)} = y^{(j)}$ | $\max(0, \alpha_i + \alpha_j - C)$ | $\min(C, \alpha_i + \alpha_j)$ |

$$\alpha_j^{\text{new,clipped}} = \begin{cases} H & \text{if } \alpha_j^{\text{new}} > H \\ \alpha_j^{\text{new}} & \text{if } L \leq \alpha_j^{\text{new}} \leq H \\ L & \text{if } \alpha_j^{\text{new}} < L \end{cases}$$

Then update $\alpha_i$ from $\alpha_i^{\text{new}} = (\zeta - \alpha_j^{\text{new}} y^{(j)}) y^{(i)}$.

### 5.4 Update $b$

After updating the pair, recompute $b$ using the KKT conditions:

$$b_i = -E_i - y^{(i)} K_{ii}(\alpha_i^{\text{new}} - \alpha_i^{\text{old}}) - y^{(j)} K_{ji}(\alpha_j^{\text{new}} - \alpha_j^{\text{old}}) + b^{\text{old}}$$

Use $b_i$ if $0 < \alpha_i^{\text{new}} < C$, $b_j$ if $0 < \alpha_j^{\text{new}} < C$, or the average otherwise.

## 6. SMO Heuristics and Convergence

### 6.1 Choosing the pair $(\alpha_i, \alpha_j)$

SMO uses a two-level heuristic:

**Outer loop — choose $\alpha_i$:**
- First pass: iterate over all training examples looking for KKT violators
- Subsequent passes: alternate between all examples and non-bound examples ($0 < \alpha_i < C$)
  (non-bound points are most likely to move and cause larger $W$ improvements)

**Inner loop — choose $\alpha_j$:**
- Select $j$ to **maximise** $|E_i - E_j|$ — this heuristic tends to give the largest step
- Fallback: scan all non-bound, then all, if no improvement found

### 6.2 KKT convergence test

A point satisfies the KKT conditions (within tolerance $\varepsilon$) iff:

$$\begin{cases}
\alpha_i = 0 & \Rightarrow & y^{(i)} h(x^{(i)}) \geq 1 - \varepsilon \\
0 < \alpha_i < C & \Rightarrow & |y^{(i)} h(x^{(i)}) - 1| \leq \varepsilon \\
\alpha_i = C & \Rightarrow & y^{(i)} h(x^{(i)}) \leq 1 + \varepsilon
\end{cases}$$

SMO terminates when all training examples satisfy KKT to within $\varepsilon$ (typically $10^{-3}$).

### 6.3 Computational complexity

| Step | Cost |
|---|---|
| One pair update | $O(n)$ (update errors $E_i$) |
| Full pass over training set | $O(n^2)$ |
| Total (empirically) | $O(n^2)$–$O(n^3)$ — much better than generic QP for sparse support sets |

In [ ]:
import numpy as np

def smo(X, y, C=1.0, kernel='linear', tol=1e-3, max_passes=50, gamma=1.0):
    n = len(y)
    alpha = np.zeros(n)
    b = 0.0
    passes = 0

    if kernel == 'linear':
        K = X @ X.T
    else:  # rbf
        diff = X[:, None, :] - X[None, :, :]
        K = np.exp(-gamma * np.sum(diff**2, axis=-1))

    def h(i):
        return float((alpha * y) @ K[:, i]) + b

    while passes < max_passes:
        num_changed = 0
        for i in range(n):
            Ei = h(i) - y[i]
            ri = y[i] * Ei
            # KKT violation check
            if not ((ri < -tol and alpha[i] < C) or (ri > tol and alpha[i] > 0)):
                continue
            # choose j != i to maximise |Ei - Ej|
            errors = np.array([h(k) - y[k] for k in range(n)])
            j = np.argmax(np.abs(Ei - errors))
            if j == i:
                j = (i + 1) % n
            Ej = errors[j]
            ai_old, aj_old = alpha[i], alpha[j]
            # compute L, H
            if y[i] != y[j]:
                L = max(0, alpha[j] - alpha[i])
                H = min(C, C + alpha[j] - alpha[i])
            else:
                L = max(0, alpha[i] + alpha[j] - C)
                H = min(C, alpha[i] + alpha[j])
            if abs(H - L) < 1e-12:
                continue
            eta = K[i, i] + K[j, j] - 2 * K[i, j]
            if eta <= 0:
                continue
            # update alpha_j
            alpha[j] += y[j] * (Ei - Ej) / eta
            alpha[j] = np.clip(alpha[j], L, H)
            if abs(alpha[j] - aj_old) < 1e-5:
                continue
            # update alpha_i
            alpha[i] += y[i] * y[j] * (aj_old - alpha[j])
            # update b
            bi = (b - Ei
                  - y[i]*(alpha[i]-ai_old)*K[i,i]
                  - y[j]*(alpha[j]-aj_old)*K[i,j])
            bj = (b - Ej
                  - y[i]*(alpha[i]-ai_old)*K[i,j]
                  - y[j]*(alpha[j]-aj_old)*K[j,j])
            if 0 < alpha[i] < C:
                b = bi
            elif 0 < alpha[j] < C:
                b = bj
            else:
                b = (bi + bj) / 2
            num_changed += 1
        passes = passes + 1 if num_changed == 0 else 0

    return alpha, b

# ── Test on linearly separable data ───────────────────────────────────────
import matplotlib.pyplot as plt
from sklearn.svm import SVC

rng = np.random.default_rng(42)
pos = rng.normal(loc=[3.5, 3.5], scale=0.5, size=(12, 2))
neg = rng.normal(loc=[1.0, 1.0], scale=0.5, size=(12, 2))
X = np.vstack([pos, neg])
y = np.hstack([np.ones(12), -np.ones(12)])

alpha_smo, b_smo = smo(X, y, C=1.0)
w_smo = (alpha_smo * y) @ X

clf = SVC(kernel='linear', C=1.0)
clf.fit(X, y)
w_sk = clf.coef_[0]
b_sk = clf.intercept_[0]

print('=== SMO vs sklearn comparison ===')
print(f'  SMO    w = {w_smo.round(4)},  b = {b_smo:.4f}')
print(f'  sklearn w = {w_sk.round(4)},  b = {b_sk:.4f}')
print(f'  ||w_smo|| = {np.linalg.norm(w_smo):.4f}')
print(f'  ||w_sk || = {np.linalg.norm(w_sk):.4f}')
print(f'  # support vectors (SMO): {(alpha_smo > 1e-4).sum()}')
print(f'  # support vectors (sklearn): {len(clf.support_vectors_)}')

# ── Visualise ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, (w, b, title) in zip(axes, [
    (w_smo, b_smo, 'SMO (our implementation)'),
    (w_sk, b_sk,   'sklearn SVC'),
]):
    xr = np.linspace(-0.5, 5.0, 300)
    ax.plot(xr, -(w[0]*xr+b)/w[1], 'k-', lw=2.2, label='boundary')
    ax.plot(xr, -(w[0]*xr+b-1)/w[1], 'b--', lw=1.5)
    ax.plot(xr, -(w[0]*xr+b+1)/w[1], 'r--', lw=1.5)
    ax.fill_between(xr, -(w[0]*xr+b-1)/w[1], -(w[0]*xr+b+1)/w[1],
                    alpha=0.10, color='#4dac26')
    ax.scatter(pos[:,0], pos[:,1], marker='x', s=80, c='#2166ac', linewidths=2, label='$y=+1$')
    ax.scatter(neg[:,0], neg[:,1], marker='o', s=65, facecolors='none',
               edgecolors='#d6604d', linewidths=2, label='$y=-1$')
    # highlight support vectors from alpha
    sv_mask = alpha_smo > 1e-4
    ax.scatter(X[sv_mask,0], X[sv_mask,1], s=260, facecolors='none',
               edgecolors='#ff7f00', linewidths=2.2, zorder=5, label='support vectors')
    ax.set_xlim(-0.5, 5.0); ax.set_ylim(-0.5, 5.0)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)

fig.suptitle('SMO Implementation vs sklearn SVC — Linear Kernel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def smo_with_history(X, y, C=1.0, tol=1e-3, max_passes=50):
    n = len(y)
    alpha = np.zeros(n)
    b = 0.0
    K = X @ X.T
    history = []

    def W():
        return np.sum(alpha) - 0.5 * np.sum(
            (alpha[:,None]*alpha[None,:]) * (y[:,None]*y[None,:]) * K
        )

    def h(i):
        return float((alpha * y) @ K[:, i]) + b

    passes = 0
    iteration = 0
    while passes < max_passes and iteration < 500:
        num_changed = 0
        for i in range(n):
            Ei = h(i) - y[i]
            ri = y[i] * Ei
            if not ((ri < -tol and alpha[i] < C) or (ri > tol and alpha[i] > 0)):
                continue
            errors = np.array([h(k) - y[k] for k in range(n)])
            j = np.argmax(np.abs(Ei - errors))
            if j == i: j = (i+1) % n
            Ej = errors[j]
            ai_old, aj_old = alpha[i], alpha[j]
            L = max(0, alpha[j]-alpha[i]) if y[i]!=y[j] else max(0, alpha[i]+alpha[j]-C)
            H = min(C, C+alpha[j]-alpha[i]) if y[i]!=y[j] else min(C, alpha[i]+alpha[j])
            if abs(H-L) < 1e-12: continue
            eta = K[i,i] + K[j,j] - 2*K[i,j]
            if eta <= 0: continue
            alpha[j] = np.clip(alpha[j] + y[j]*(Ei-Ej)/eta, L, H)
            if abs(alpha[j]-aj_old) < 1e-5: continue
            alpha[i] += y[i]*y[j]*(aj_old - alpha[j])
            bi = b - Ei - y[i]*(alpha[i]-ai_old)*K[i,i] - y[j]*(alpha[j]-aj_old)*K[i,j]
            bj = b - Ej - y[i]*(alpha[i]-ai_old)*K[i,j] - y[j]*(alpha[j]-aj_old)*K[j,j]
            b = bi if 0<alpha[i]<C else (bj if 0<alpha[j]<C else (bi+bj)/2)
            num_changed += 1
            iteration += 1
            history.append({'iter': iteration, 'W': W(),
                             'n_sv': (alpha > 1e-4).sum(),
                             'i': i, 'j': j})
        passes = passes + 1 if num_changed == 0 else 0
    return alpha, b, history

rng = np.random.default_rng(42)
pos = rng.normal(loc=[3.0, 3.0], scale=0.6, size=(10, 2))
neg = rng.normal(loc=[0.8, 0.8], scale=0.6, size=(10, 2))
X = np.vstack([pos, neg])
y = np.hstack([np.ones(10), -np.ones(10)])

alpha_h, b_h, history = smo_with_history(X, y, C=1.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.0))

iters = [h['iter'] for h in history]
Ws    = [h['W'] for h in history]
n_svs = [h['n_sv'] for h in history]

ax = axes[0]
ax.plot(iters, Ws, 'b-', lw=2)
ax.set_xlabel('SMO pair-update iteration', fontsize=10)
ax.set_ylabel('Dual objective $W(\\alpha)$', fontsize=10)
ax.set_title('SMO Convergence: Dual Objective vs Iteration', fontsize=11)
ax.text(0.97, 0.05, f'Final $W={Ws[-1]:.4f}$\n{len(history)} updates total',
        transform=ax.transAxes, ha='right', fontsize=9,
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

ax = axes[1]
ax.plot(iters, n_svs, 'r-', lw=2)
ax.axhline(n_svs[-1], color='k', ls='--', lw=1.5)
ax.set_xlabel('SMO pair-update iteration', fontsize=10)
ax.set_ylabel('Number of support vectors', fontsize=10)
ax.set_title('Support Vector Count vs Iteration', fontsize=11)
ax.text(0.97, 0.05, f'Final: {n_svs[-1]} SVs',
        transform=ax.transAxes, ha='right', fontsize=9,
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

fig.suptitle('SMO Algorithm Convergence', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

| Concept | Formula / Description | Role |
|---|---|---|
| Slack variable | $\xi_i \geq 0$ | Allows margin violation for point $i$ |
| Soft-margin primal | $\min \frac{1}{2}\|w\|^2 + C\sum\xi_i$ s.t. $y^{(i)}(w^Tx^{(i)}+b)\geq 1-\xi_i$ | Regularised SVM |
| $C$ parameter | Large $C$ → hard margin; small $C$ → soft/wide margin | Regularisation |
| Soft-margin dual | Same as hard-margin but $0\leq\alpha_i\leq C$ | Solvable by SMO |
| KKT categories | $\alpha_i=0$: outside; $0<\alpha_i<C$: on margin; $\alpha_i=C$: inside/error | Classify each point |
| SMO | Optimise pairs $(\alpha_i,\alpha_j)$ with closed-form update | Efficient solver |
| SMO update | $\alpha_j^{\text{new}} = \alpha_j^{\text{old}} - y^{(j)}(E_i-E_j)/\eta$, clipped to $[L,H]$ | Closed form |
| KKT convergence | Check all three KKT conditions to $\pm\varepsilon$ | Termination |

**Where SMO fits in the full SVM story:**

```
Soft-margin primal (xi_i, C)
    |  Lagrange duality  (see ml_001_15_4)
    v
Soft-margin dual: max W(alpha), 0 <= alpha_i <= C
    |  SMO algorithm
    v
alpha* -> support vectors -> w*, b*  -> predictions
```

**Key insight:** SMO decomposes the $n$-variable QP into a sequence of 2-variable QPs each with
a closed-form solution. This, combined with KKT-based convergence testing, makes training
large-scale SVMs practical — it is the standard algorithm in scikit-learn's `SVC`.